Shinozaki Lab. 2020, 2026

In [ ]:
import os

# Google Colab環境かどうかを判定
if 'google.colab' in str(get_ipython()):
    # リポジトリがまだなければクローン
    if not os.path.exists('speech-information-technology'):
        !git clone https://github.com/tttslab/speech-information-technology.git

    # 作業ディレクトリをvoicechangerに移動
    # これにより、'samplewav/xxx.wav' という相対パスがそのまま通るようになります
    os.chdir('speech-information-technology/convolution')
    print(f"Current directory: {os.getcwd()}")

In [ ]:
# Install needed library.
!pip install scipy numpy

In [ ]:
import numpy as np
from scipy.io.wavfile import write, read
import IPython.display as ipd

### 1. Define convolution function

Given signal $x$ with length N and $h$ with length M, we can calculate convolution using the formula below:

$(x*h)[n]=\sum_{k=0}^{N-1} x[k]h[n-k]=\sum_{k=0}^{M-1}h[k]x[n-k]$

In [ ]:
def convolution(x, h):
    '''
    Calculate convolution of x and h.
    ::x:: array-like with length N
    ::h:: array-like with length M
    ------------------------
    ::out:: array-like with length (M+N-1)
    '''
    x, h = np.array(x, dtype=np.int32), np.array(h, dtype=np.int32)
    assert x.ndim == h.ndim == 1
    N, M = x.shape[0], h.shape[0]

    out = np.zeros(M+N-1, dtype=np.int32)

    # Version 1
    for n in range(0, M+N-1):
        for k in range(M):
            if 0 <= n - k < N:
                out[n] += h[k] * x[n-k]

#     # Version 2
#     # Step 1: padding 0 before x
#     x_pad = np.concatenate([np.zeros(M-1, dtype=np.int32), x], axis=0)
#     # Step 2: reverse h and padding 0 after h_reverse
#     h_reverse = h[::-1]
#     h_pad = np.concatenate([h_reverse, np.zeros(N-1, dtype=np.int32)], axis=0)
#     # Step 3: shift and convolve
#     for n in range(0, M+N-1):
#         out[n] = sum(np.roll(h_pad, shift=n) * x_pad)
    return out

In [ ]:
# Define input arrays
x = [1, 2, 3, 4, 5]
h = [8, 4, 2, 1]

# Use self-defined function
y = convolution(x, h)
print("Output of convolution function:", y)

# Use convolve function provided by numpy
y2 = np.convolve(x, h)
print("Output of numpy.convolve:", y2)

# Assert they output same result
assert (y == y2).all()

### 2. Load wavs

In [ ]:
# Define paths
wav_path_1 = "./bgm_maoudamashii_neorock76_mono48k_sub.wav" # path to wav file 1
wav_path_2 = "./can-impulse-response_2_mono48k.wav" # path to wav file 2
save_path = "./conv.wav" # path to save result


sample_rate_1, wav_1 = read(wav_path_1)
sample_rate_2, wav_2 = read(wav_path_2)
assert sample_rate_1 == sample_rate_2, 'Sample rates of wav files are not consistent.'

print("wav_1:")
display(ipd.Audio(wav_1, rate=sample_rate_1))
print("wav_2:")
display(ipd.Audio(wav_2, rate=sample_rate_2))

### 3. Compute convolution

In [ ]:
conv = np.convolve(wav_1.astype(np.int64), wav_2.astype(np.int64))

print("Output wav:")
display(ipd.Audio(conv, rate=sample_rate_1))

### 4. Save output wav

In [ ]:
# Normalize the result to 16 bit integer.
conv = np.array(conv / np.max(np.abs(conv)) * 32767, dtype=np.int16)

write(filename=save_path, rate=sample_rate_1, data=conv)
print("Output saved to:", save_path)